In [13]:
%pip install seaborn
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 15.7 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import os


In [6]:
# DOWNLOAD shared google drive into your local device
# Find the folder in your Library/Finder and click copy or copy path
# paste it into file_path

# Drive:
# file_path: '/content/drive/MyDrive/COGS_189_EEG_Data'

file_path = '/Users/earanda/Downloads/COGS_189_EEG_Data'
 
all_X      = []   # will hold (trials, 8, 225) arrays
all_y      = []   # will hold (trials,) binary label arrays
trial_log  = []   # metadata: sub / ses / run / n_trials / n_errors
 
for sub in sorted(os.listdir(file_path)):
    sub_path = os.path.join(file_path, sub)
    if not (os.path.isdir(sub_path) and sub.startswith('sub-')):
        continue
 
    for ses in sorted(os.listdir(sub_path)):
        ses_path = os.path.join(sub_path, ses)
        if not (os.path.isdir(ses_path) and ses.startswith('ses-')):
            continue
 
        # Collect all run numbers present in this session
        run_nums = set()
        for fname in os.listdir(ses_path):
            if fname.startswith('eeg_trials_run-') and fname.endswith('.npy'):
                run_num = fname.replace('eeg_trials_run-', '').replace('.npy', '')
                run_nums.add(run_num)
 
        for run_num in sorted(run_nums):
            trials_path = os.path.join(ses_path, f'eeg_trials_run-{run_num}.npy')
            labels_path = os.path.join(ses_path, f'labels_run-{run_num}.npy')
 
            # Only load if both files exist
            if not (os.path.exists(trials_path) and os.path.exists(labels_path)):
                print(f"  Skipping {sub}/{ses}/run-{run_num} — missing file")
                continue
 
            X_run = np.load(trials_path, allow_pickle=True)   # (trials, 8, 225)
            y_str = np.load(labels_path, allow_pickle=True)   # (trials,) string
 
            # Handle label shape mismatch (trial_metadata has 50 rows, trials 49)
            # Use the shorter length to keep X and y aligned
            n = min(X_run.shape[0], y_str.shape[0])
            X_run = X_run[:n]
            y_str = y_str[:n]
 
            # Binary encode: error=1, correct=0  (A2 Cell 18 convention)
            y_run = (y_str == 'error').astype(int)
 
            n_err = int(np.sum(y_run))
            n_cor = int(np.sum(y_run == 0))
 
            all_X.append(X_run)
            all_y.append(y_run)
            trial_log.append({
                'sub': sub, 'ses': ses, 'run': run_num,
                'n_trials': n, 'n_errors': n_err, 'n_correct': n_cor
            })
            print(f"  Loaded {sub}/{ses}/run-{run_num}: "
                  f"{n} trials  ({n_cor} correct, {n_err} error)")
 
# concat all the runs

# (total_trials, 8, 225)
X_all = np.concatenate(all_X, axis=0) 
# (total_trials,)
y_all = np.concatenate(all_y, axis=0)  
 
total_trials = X_all.shape[0]
total_errors = int(np.sum(y_all))
total_correct = int(np.sum(y_all == 0))
 
print(f"  Total runs loaded: {len(trial_log)}")
print(f"  Total trials: {total_trials}")
print(f"  Total correct: {total_correct}  ({100*total_correct/total_trials:.1f}%)")
print(f"  Total errors: {total_errors}   ({100*total_errors/total_trials:.1f}%)")
print(f"  X shape: {X_all.shape}")
print(f"  y shape: {y_all.shape}")
 
#parameters for dataset
fs               = 250.0       # Hz
dt               = 1000. / fs  # ms (A1 Cell 39)
sdt              = int(round(dt))
epoch_start_ms   = -500
epoch_end_ms     =  400
pre_start_ms     = -500
pre_end_ms       =   0
baseline_start_ms = -500
baseline_end_ms   = -400
 
pre_s = np.round((pre_start_ms  - epoch_start_ms) / sdt).astype(int)   # 0
pre_e = np.round((pre_end_ms    - epoch_start_ms) / sdt).astype(int)   # 125
bl_s  = np.round((baseline_start_ms - epoch_start_ms) / sdt).astype(int)
bl_e  = np.round((baseline_end_ms   - epoch_start_ms) / sdt).astype(int)
 
channel_names = ['Fz','F3','Cz','C3','C4','Pz','O1','O2']
times_ms = np.linspace(epoch_start_ms, epoch_end_ms, X_all.shape[2])
 
err_idx = np.where(y_all == 1)[0]
cor_idx = np.where(y_all == 0)[0]


  Loaded sub-01/ses-01/run-01: 49 trials  (43 correct, 6 error)
  Loaded sub-01/ses-01/run-02: 50 trials  (44 correct, 6 error)
  Loaded sub-01/ses-01/run-03: 50 trials  (44 correct, 6 error)
  Loaded sub-01/ses-01/run-04: 50 trials  (43 correct, 7 error)
  Loaded sub-02/ses-01/run-01: 50 trials  (48 correct, 2 error)
  Loaded sub-02/ses-01/run-02: 50 trials  (45 correct, 5 error)
  Loaded sub-02/ses-01/run-03: 50 trials  (44 correct, 6 error)
  Loaded sub-02/ses-01/run-04: 50 trials  (46 correct, 4 error)
  Loaded sub-02/ses-02/run-01: 50 trials  (44 correct, 6 error)
  Loaded sub-02/ses-02/run-02: 50 trials  (44 correct, 6 error)
  Loaded sub-02/ses-02/run-03: 50 trials  (46 correct, 4 error)
  Loaded sub-02/ses-02/run-04: 50 trials  (49 correct, 1 error)
  Loaded sub-02/ses-02/run-05: 50 trials  (45 correct, 5 error)
  Loaded sub-02/ses-02/run-06: 50 trials  (45 correct, 5 error)
  Loaded sub-02/ses-02/run-07: 49 trials  (46 correct, 3 error)
  Loaded sub-02/ses-02/run-08: 50 trials

In [ ]:
# baseline correction for 3D EEG data
# takes value and subtracts it by the mean of all X values, to normalize it around 0
X_bl = X_all.copy()
for i in range(X_bl.shape[0]):              
    for ch in range(X_bl.shape[1]):
        X_bl[i, ch, :] = X_bl[i, ch, :] - np.mean(X_bl[i, ch, bl_s:bl_e])

print(f"  Baseline corrected: {X_bl.shape}")

  Baseline corrected: (798, 8, 225)
